<big><big>001: prelude</big></big>
<a id='index'></a><br>


## 🍏 AppleCider (Applying multimodal Learning to Classify transient Detections Early)* 🍏
<small><small><i>*still not committed to all the word chocies...multimodal Learning... machine learning.. i am open to swapping.</small></small></small>
<br><br>
    

    
<u>notebook outline:</u>
- [3 modality dataset vs 4 modality dataset](#three)
- [more about the data](#dataset) (cartoon is here)
    - [ZTF alerts (includes example alert)](#alert) 
    - [spectra](#spectra)
- [dataset (not included)](#mini)
    - [preprocess dataset](#process)
- [AppleCider](#drink)
    - [config](#config)
    - [DataGenerator](#datagen)
    - [plot things](#plot)
    - [drink AppleCider (run the model)](#drink2)
<br><br>


***
<a id='three'></a><br>
general things about AppleCider / AppleCider Dataset
- AppleCider started out with photometry, images, metadata. these modalities are extracted from [ZTF alert packets](https://zwickytransientfacility.github.io/ztf-avro-alert/schema.html) (also see [Patterson et al 2019](https://iopscience.iop.org/article/10.1088/1538-3873/aae904)) from the alert broker [Kowalski](https://github.com/skyportal/kowalski). for reference, ZTF can get ~ 1 million alerts per night.
<br>
<br>
    
<div>
<img src="./img/dataset.png" width="1150"/>
</div>
    

- there are <b>12093 objects that have spectra AND are in the 10 classes that we are <s>currently</s> interested in classifying</b>. these spectra are not all from the same instrument.
    - spectra comes from 4 places: [Fritz](https://github.com/fritz-marshal/fritz), SDSS, DESI, and Yu-Jing @ Caltech (s/o Yu-jing. the spectra marked as "WIS" is from him but includes spectra from WISeREP, TNS, and the old Growth Marshal).
        - a little under 7,000 of the objects with spectra are from [SEDM](https://www.ztf.caltech.edu/ztf-sedm.html) (see `SEDM_BrightTransientSurveyt.csv`)

<br><br>

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import pickle
import random
import joblib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import sys ; sys.path.insert(0, '/AppleCider')

from AppleCider.preprocess.data_preprocessor import AlertProcessor, PhotometryProcessor, DataPreprocessor, SpectraProcessor
from AppleCider.preprocess.data_preprocessor import DataSorter
from AppleCider.preprocess.transient_dataset import TransientDataset

import AppleCider.preprocess.plot_data as plot_data
from AppleCider.preprocess.plot_data import plot_image
from AppleCider.preprocess.plot_data import plot_dataset
import AppleCider.core.dataset as dataset
from AppleCider.core.dataset import DataGenerator
import AppleCider.core.drink as drink

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
dataset = pd.read_csv('./csv-pkl/SEDM_BrightTransientSurvey.csv')
dataset

,obj_id,type
0,ZTF18aakpnyt,AGN
1,ZTF18adjoimi,AGN
2,ZTF18aamyexb,AGN
3,ZTF18aasxwhv,SN Ia
4,ZTF21abetapt,SN Ia
...,...,...
6938,ZTF24abbenwl,SN II
6939,ZTF24abbiqob,SN Ia
6940,ZTF24abbrcrq,SN Ia
6941,ZTF24abbsboo,SN Ia


In [3]:
CLASSES = ['SN Ia','SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event']

## remove objects not in specific class list
## there are no CVs in this BTS dataset so...
dataset = dataset[dataset['type'].isin(CLASSES)]
dataset['type'].value_counts()

type
SN Ia                     4867
SN II                      489
SN IIP                     471
Cataclysmic                272
SN IIn                     163
AGN                        112
SN Ic                      112
SN Ib                      108
SN IIb                      83
Tidal Disruption Event      34
Name: count, dtype: int64

***
##  🍏  more about the data 🍏 

<a id='alert'></a><br>



<br>


<div>
<img src="./img/AppleCider_alert_modal.png" width="1150"/>
</div>



<u>from ZTF alert packet</u>:

    photometry (converted from magnitude to flux)
    - MJD (for now, cutting off at first 10 days of an alert)
    - filters: ztf-g, ztf-r, ztf-i (many objects have no photometry in ztf-i)
    
    images:
    - science iamge: image of transient 
    - reference image: archival image 
    - difference
    
    metadata:
    - 10 features:
        - sgscore 1 & sgscore 2: star/galaxy score of nearest two PS1 source
        - distpsnr 1 & distpsnr 2: distance to nearest PS1 sources [arcsec]
        - ra: right ascension of source [deg]
        - dec: declination of source [deg]
        - sharpnr: sharp parameter of nearest source in reference
        - scorr: peak-pixel S/N in detection image
        - sky: local sky background estimate [DN]
        - nmtchps: # of PS1cross-matches within 30 arcsec

<i><small>[back to index](#index)</i></small>

an unprocesed alert looks like this:

In [ ]:
data_dir = '../AppleCider/SEDM_folder'
obj_id = 'ZTF17aabtvsy'

obj_alert_path = os.path.join(data_dir, obj_id, 'alerts.npy')
obj_alert = np.load(obj_alert_path, allow_pickle=True)

obj_alert

so, they're very long.

to actually get the photometry, metadata, images, each object's `alert.npy` is pre-processed and saved to a "new" alert. for example: `ZTF24aazjdml_alert_44.npy`. The `_44` represents the final valid index. in our case, the final valid index is where the photometry is cut off at 10 days. the higher the "valid index", the more points there are. you might notice that some processed alerts will have a `_1`. these are the processed alerts that have only two points in photometry. anything less than that, and there will be no alert saved during preprocessing. these two points can be in the same filter or in two different filters. 

In [5]:
data_dir = './csv-pkl/'
## example processed alert:
obj_id = 'ZTF17aabtvsy_alert_1.npy'

obj_ex_path = os.path.join(data_dir, obj_id) 
obj_ex = np.load(obj_ex_path, allow_pickle=True).item()
print(f"obj_id: {obj_ex['obj_id']}, type: {obj_ex['target']}")
obj_ex 

obj_id: ZTF17aabtvsy, type: SN Ia


{'obj_id': 'ZTF17aabtvsy',
 'photometry': array([[ 0.       ,  0.       ,  1.       ,  0.       ],
        [ 1.9791666,  0.       , -1.       ,  0.       ]]),
 'metadata': sgscore1       0.182345
 sgscore2       0.018833
 distpsnr1      0.413914
 distpsnr2     15.558051
 ra           158.883523
 dec           37.649664
 nmtchps       14.000000
 sharpnr        0.168000
 scorr          5.023887
 sky            0.028106
 Name: 1, dtype: float64,
 'images': array([[[ 0.01490862,  0.01522977,  0.01255172],
         [ 0.01522451,  0.01527908,  0.00541103],
         [ 0.01508745,  0.01527026,  0.01159723],
         ...,
         [ 0.01599111,  0.01513144, -0.03298366],
         [ 0.0145049 ,  0.01511197,  0.0013018 ],
         [ 0.01438295,  0.01523208,  0.01333428]],
 
        [[ 0.01531396,  0.01503041, -0.01975301],
         [ 0.01538159,  0.01512525, -0.01647759],
         [ 0.01493295,  0.01530884,  0.02226405],
         ...,
         [ 0.01587359,  0.01535224,  0.00876437],
         [ 0


<a id='spectra'></a>

<div>
<img src="../img/AppleCider_spectra.png" width="350"/>
</div>


<u>spectra (not from ZTF alerts)</u>:

unlike our beautiful ztf photometry, metadata, and images... spectra comes from a variety of instruments, databases. not all `spectra.csv` have the same columns. however, they will all have some variation of `"ZTF_id"` (obj_id), `"wavelength"`, `"flux"`, `"instrument name"` and/or `"telescope_name"`, `"observed_at_mjd"`.

if they are not from Fritz, there will be a `"source"` column (i.e. SDSS, DESI) and a `"spec_id"` column that has the database's ID for it. additionally, some spectra will have other columns like `"wdisp"`, `"sky"`. it depends. right now, all we're using is `"wavelength"`, `"flux"`. 

if you want to know more about the alert pre-processing steps, see [/notebooks/001-data-processing.ipynb](https://github.com/ajunell/AppleCider_Data/blob/main/notebooks/001-data-processing.ipynb). 


***
## 🍏 dataset (not included) 🍏 

<a id='mini'></a>
each object folder has `alerts.npy`, `photometry.csv`, `spectra.csv`. their spectra comes from several different instruments, but is mainly from SEDM. you will need to do the alert preprocessing steps on these objects!. the cells to do it are below. you can also see them here  ([/notebooks/001-data-processing.ipynb](https://github.com/ajunell/AppleCider/blob/main/notebooks/001-data-processing.ipynb)) (3/9: although currently there might be a broken cell or two)


<i><small>[back to index](#index)</i></small>

preprocess our new alerts which have photometry, metadata, images, and spectra...

In [4]:
## preprocess our new alerts which have photometry, metadata, images, and spectra
data_dir = '/AppleCider_Data/SEDM_folder'

## wherever you want to save processed alerts 
TEST_DATA_PATH = './data_test_BTS'
TRAIN_DATA_PATH = './data_train_BTS'

#train_df, test_df = train_test_split(dataset, test_size=0.10, stratify=dataset['type'])
#train_df.to_csv('./csv-pkl/train_df_BTS.csv', index=False)
#test_df.to_csv('./csv-pkl/test_df_BTS.csv', index=False)
train_df = pd.read_csv('./csv-pkl/train_df_BTS.csv')
test_df = pd.read_csv('./csv-pkl/test_df_BTS.csv')

In [5]:
train_df['type'].value_counts()

type
SN Ia                     4379
SN II                      440
SN IIP                     424
Cataclysmic                245
SN IIn                     147
AGN                        101
SN Ic                      101
SN Ib                       97
SN IIb                      75
Tidal Disruption Event      30
Name: count, dtype: int64

In [6]:
test_df['type'].value_counts()

type
SN Ia                     488
SN II                      49
SN IIP                     47
Cataclysmic                27
SN IIn                     16
SN Ic                      11
AGN                        11
SN Ib                      11
SN IIb                      8
Tidal Disruption Event      4
Name: count, dtype: int64

<a id='process'></a><br>

some of the objects will probably fail the preprocessing step requirements (i.e. not enough points in photometry during first 10 days) so don't be alarmed if you get any print statements related to this during preprocessing. if you increase `max_mjd`, more objects will pass)

originally, the model classified objects over a period of 90 days. this meant that an object in the dataset might've had more than 1 alert. because of this, there needed to be a way to make sure that alerts from the same object were either all in the training set or all in the testing set. several of the weird steps before the DataGenerator exist to do this. 

In [9]:
# make dataframe of object IDs with associated alert file names, and clasification 
data_test, data_train = DataSorter.create_df_of_object_alerts_in_dataset(test_df, train_df, TEST_DATA_PATH, TRAIN_DATA_PATH)
data_train

,name,file,type
0,ZTF17aaajnki,ZTF17aaajnki_alert_1.npy,AGN
1,ZTF17aaaobyl,ZTF17aaaobyl_alert_3.npy,Cataclysmic
2,ZTF17aaaukqn,ZTF17aaaukqn_alert_20.npy,Cataclysmic
3,ZTF17aaawgkc,ZTF17aaawgkc_alert_1.npy,Cataclysmic
4,ZTF17aabtvsy,ZTF17aabtvsy_alert_1.npy,SN Ia
...,...,...,...
5669,ZTF24abbbdqs,ZTF24abbbdqs_alert_11.npy,SN Ia
5670,ZTF24abbdtfp,ZTF24abbdtfp_alert_11.npy,SN Ia
5671,ZTF24abbenwl,ZTF24abbenwl_alert_10.npy,SN II
5672,ZTF24abbiqob,ZTF24abbiqob_alert_11.npy,SN Ia


In [10]:
#data_train.to_csv('./csv-pkl/data_train_BTS.csv', index=False)
#data_test.to_csv('./csv-pkl/data_test_BTS.csv', index=False)

data_train = pd.read_csv('./csv-pkl/data_train_BTS.csv')
data_test = pd.read_csv('./csv-pkl/data_test_BTS.csv')

***
# 🍏  drink AppleCider 🍏 

<a id='drink'></a><br>

<img src="./img/applecider-cartoon.png" width="1150"/>
</div>

(spectra dimensions actually 1 x 3481 but my photoshop trial expired so please forgive me </3)

- photometry: informer from [Zhou 2020](https://arxiv.org/abs/2012.07436) / [AstroM3](https://arxiv.org/abs/2411.08842), [github](https://github.com/MeriDK/AstroM3/tree/main)
- images: neural net from [BTSbot](https://arxiv.org/abs/2401.15167), [github](https://github.com/nabeelre/BTSbot)
- metadata: perceptron from [AstroM3](https://arxiv.org/abs/2411.08842), [github](https://github.com/MeriDK/AstroM3/tree/main)
- spectra: neural net from [Wu 2024](https://academic.oup.com/mnras/article/527/1/1163/7283157) / [AstroM3](https://arxiv.org/abs/2411.08842), [github](https://github.com/MeriDK/AstroM3/tree/main)

<br><br>
<i><small>[back to index](#index)</i></small>

In [11]:
def split_and_compute_class_weights(df, max_samples, step, group_labels=False, save_files=True, save_train_files_path= None, save_val_files_path=None, save_class_weights_path = None, split_ratio=0.8, random_seed=42, nb=None, verbose=False):
    
    """ create train files, val files + class weight dictionary and save them
        note: normally compute_class_weight would have balanced class weights, but since 
          i'm using a smaller dataset with public data -> class_weight=None

    """
    
    if group_labels:
        group_labels = {'SN Ia': 0, 'SN Ic': 0, 'SN Ib': 0, 'SN II': 1, 'SN IIP': 1, 'SN IIn': 1,
                        'SN IIb': 1, 'Cataclysmic': 2, 'AGN': 3, 'Tidal Disruption Event': 4}
        df.replace({step:group_labels})
   
    else:
        id2target = {'SN Ia':0 ,'SN Ic':1,  'SN Ib':2 , 'SN II': 3, 'SN IIP': 4, 'SN IIn': 5,
                    'SN IIb': 6, 'Cataclysmic': 7, 'AGN': 8, 'Tidal Disruption Event': 9}
        target2id = {v: k for k, v in id2target.items()}
        
        df = df.replace({step: id2target})
    
    train_df_list, val_df_list = [], []

    for cls in df[step].unique():
        df_cls = df[df[step] == cls]
        df_not_cls = df[df[step] != cls]

        if len(df_cls) > max_samples:
            print(f'Down sampled class {cls} from {len(df_cls)} to {max_samples}')
            df_cls_down = df_cls.sample(n=max_samples, random_state=random_seed)
            df = pd.concat([df_not_cls, df_cls_down], ignore_index=True)
    
    unique_labels = df[step].unique()
    
    for label in unique_labels:
        df_filtered = df[df[step] == label]
        unique_obj_ids = df_filtered['name'].unique()
        random.seed(random_seed)
        random.shuffle(unique_obj_ids)
        split_idx = int(len(unique_obj_ids) * split_ratio)
        train_obj_ids = unique_obj_ids[:split_idx]
        val_obj_ids = unique_obj_ids[split_idx:]
        train_df_list.append(df_filtered[df_filtered['name'].isin(train_obj_ids)])
        val_df_list.append(df_filtered[df_filtered['name'].isin(val_obj_ids)])
    
    train_df = pd.concat(train_df_list).reset_index(drop=True)
    val_df = pd.concat(val_df_list).reset_index(drop=True)

    train_obj_ids = train_df['name'].unique()
    val_obj_ids = val_df['name'].unique()

    assert len(set(train_obj_ids).intersection(set(val_obj_ids))) == 0

    class_weights = compute_class_weight(class_weight='balanced', classes=unique_labels, y=train_df[step])
    class_weight_dict = dict(zip(unique_labels, class_weights))
    
    train_files = train_df['file'].tolist()
    val_files = val_df['file'].tolist()
   
    if save_files:
        with open(os.path.join(save_train_files_path), 'wb') as file:
            pickle.dump(train_files, file)
            print(f"saved train files to {save_train_files_path}.")  
        with open(os.path.join(save_val_files_path), 'wb') as file:
            pickle.dump(val_files, file)
            print(f"saved val files to {save_val_files_path}.")
        with open(os.path.join(save_class_weights_path), 'wb') as file:
            pickle.dump(class_weight_dict, file)
            print(f"saved weights to {save_class_weights_path}.")
    
    return train_files, val_files, class_weight_dict


In [ ]:
## this is a another relic of the "more than one alert" per object thing
## this will save .pkl for train_files, val_files and class weights 
train, val, class_weight = split_and_compute_class_weights(data_train, 300,'type', group_labels=False, save_files=True,
                                    save_train_files_path= './csv-pkl/train_files_BTS.pkl',
                                    save_val_files_path='./csv-pkl/val_files_BTS.pkl',
                                    save_class_weights_path = './csv-pkl/class_weights_BTS.pkl',
                                    split_ratio=0.8, random_seed=42, nb=None, verbose=False)

In [ ]:
## eventually there will be a more elegant way to do this.... but this works for now! 

class Metadata(torch.utils.data.Dataset):

    def __init__(self, preprocessed_path, df, file_list=None, **kwargs):
        super().__init__(**kwargs)
        self.preprocessed_path = preprocessed_path
        self.df = df

        if file_list is not None:
            self.data_files = file_list
        else:
            self.data_files = [f for f in os.listdir(preprocessed_path) if f.endswith('.npy')]
        
    def __len__(self): 
        return(len(self.data_files))
    
    def __getitem__(self, index):    
        ''' load processed object alerts to get photometry, metadata, images''' 
        file_path = os.path.join(self.preprocessed_path, str(self.data_files[index]))
        sample = np.load(file_path, allow_pickle=True).item()

        metadata = sample['metadata'].to_numpy()

        return metadata

from sklearn.preprocessing import StandardScaler
def get_metadata_scaler(metadata, metadata_scaler_path):
    
    metadata = [el for el in metadata]
    metadata = np.asarray(metadata, dtype=object)
    
    scaler = StandardScaler()
    scaler.fit(metadata)
    
    print("Column means:", scaler.mean_)
    print("Column standard deviations:", np.sqrt(scaler.var_),"\n")
    
    print("save to:", metadata_scaler_path)
    joblib.dump(scaler, os.path.join(metadata_scaler_path,'scaler_BTS.pkl'))


In [16]:
TRAIN_DATA_PATH = './data_train_BTS'

filen_train = os.path.join('./csv-pkl/', 'train_files_BTS.pkl')
with open(filen_train, 'rb') as file:
    train_files= pickle.load(file)
    
data_train = pd.read_csv('./csv-pkl/data_train_BTS.csv')

metadata_dataset = Metadata(TRAIN_DATA_PATH, data_train,
                            train_files)

metadata_scaler_path = './csv-pkl/'
get_metadata_scaler(metadata_dataset, metadata_scaler_path)

Column means: [-1.09093091e+00 -8.84782283e+00  3.14910544e+00 -2.43840155e-01
  1.85708662e+02  2.31804157e+01  9.07785620e+00  2.79418174e-01
  2.48344062e+01  1.00347523e-01]
Column standard deviations: [ 36.3266212   95.74348454  26.07618319  96.77310398 102.04271462
  25.71408357  27.26985146   0.26464672  22.02453472   2.30819406] 

save to: ./csv-pkl/


<a id='config'></a><br>

In [41]:
config = {
        
        'project': 'AppleCider_BTS',
        'mode': 'all',   # 'clip', 'photo', 'spectra', 'meta', 'image', 'ztf', 'all'
        'config_from': False,
        'random_seed': 42,   # 42, 66, 0, 12, 123
        'use_wandb': False, 
        'save_weights': True,
        'weights_path': './weights_BTS',
        
        'freeze': False,
        'use_pretrain': False,
        
        ## data paths
        'preprocessed_path': './data_train_BTS',  ## your processed data path for training set goes here
        'df_path': './csv-pkl/data_train_BTS.csv',
        'train_files_path': './csv-pkl/train_files_BTS.pkl',
        'val_files_path': './csv-pkl/val_files_BTS.pkl',
        'class_weights_path': './csv-pkl/class_weights_BTS.pkl',
        'class_weights': True,
        'generate_train_val_files': False,  # if you haven't already made the files, or use the ones i did,
                                            # you can also generate them this way... but you will need to 
                                            # still make the metadata scaler above 
        
        'step': 'type',
        'classes': ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event'],
        'group_labels': False,
        'max_samples': 5000, ## you should downsample! i am not because example
        'num_classes': 10,
        'seq_len': 275,
        
        ## if 'use_wandb': False AND 'save_weights': True,
        ## file will be: f'weights-{datetime.now().strftime("%Y-%m-%d-%H-%M")}-{epoch}-.pth'"
        ## if you want more specific file names: use 'custom_weight_path' : True
        ## file will be: f'weights-{datetime.now().strftime("%Y-%m-%d-%H-%M")}-{epoch}-{self.custom_weight_name}.pth'
        'custom_weight_path': False,
        'custom_weight_name': 'BTS_public_trial',
        
        ## photometry model
        'p_enc_in': 4,
        'p_d_model': 12,
        'p_dropout': 0.2,
        'p_factor': 1,
        'p_output_attention': False,
        'p_n_heads': 8,
        'p_d_ff': 64,     # usually 512 but notebook is a quick example so 64 it is 
        'p_activation': 'gelu',
        'p_e_layers': 8,
        
        ## spectra model
        's_dropout': 0.2,
        's_conv_channels': [1, 64, 64, 32, 32],
        's_kernel_size': 3,
        's_mp_kernel_size': 4,
        
        ## metadata model
        'm_hidden_dim': 64, # usually 512 but notebook is a quick example so 64 it is 
        'm_dropout': 0.2,
        'meta_cols': range(0, 10),
        'scaler_path': './csv-pkl/scaler_BTS.pkl',

        ## image model
        'input_channels': 3,
        'conv1_channels': 64,
        'conv2_channels': 16,
        'conv_kernel': 3,
        'conv_dropout1': 0.45,
        'conv_dropout2': 0.65,
        
        ## multimodal model
        'hidden_dim': 64, # usually 512 but notebook is a quick example so 64 it is 
        'fusion': 'avg',    # 'avg', 'concat'
        
        'batch_size': 32,
        'lr': 0.001,
        'beta1': 0.9,
        'beta2': 0.999,
        'weight_decay': 0.01,
        'epochs': 2,
        'early_stopping_patience': 4,
        'scheduler': 'ReduceLROnPlateau',  # 'ExponentialLR', 'ReduceLROnPlateau'
        'gamma': 0.9,  # for ExponentialLR scheduler
        'factor': 0.3,   # for ReduceLROnPlateau scheduler
        'patience': 3,    # for ReduceLROnPlateau scheduler
        'warmup': False,
        'warmup_epochs': 10,
        'clip_grad': False,
        'clip_value': 5}

In [ ]:
<a id='datagen'></a><br>

In [42]:
train_dataset = DataGenerator(config=config, split='train')
val_dataset = DataGenerator(config=config, split='val')

<a id='plot'></a><br>

In [ ]:
## use this if you want to know the object id, type and alert file
plot_dataset.plot_dataset_item_named(train_dataset, 2, joblib.load('./csv-pkl/train_files_BTS.pkl'))

In [ ]:
## if you don't care about knowing the object id... you can use this 
for i in range(12,15):
    plot_dataset.plot_dataset_item(train_dataset, i)

In [45]:
train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], collate_fn = drink.collate_func,shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=config['batch_size'], collate_fn = drink.collate_func, shuffle=True)

In [46]:
batch = next(iter(train_dataloader))
photometry, photometry_mask, metadata, images, spectra, target  = batch

In [47]:
photometry.shape, photometry_mask.shape, metadata.shape, images.shape, spectra.shape, target.shape

(torch.Size([32, 275, 4]),
 torch.Size([32, 275]),
 torch.Size([32, 10]),
 torch.Size([32, 3, 63, 63]),
 torch.Size([32, 1, 3481]),
 torch.Size([32]))

<a id='drink2'></a>
you can run the model like this! 


<i><small>[back to index](#index)</i></small>

In [ ]:
drink.run(config)